# Agronavis knowledge base for Sahayak

Online, Sahayak answers from an open model **plus the notes this notebook builds**, so a Bardhaman farmer gets Bengal advice rather than whatever the model read on the internet.

**Sources**
1. **Kisan Call Centre (KCC)**: real farmer questions answered by the government's farm tele-advisors, by state, crop and season. Government data under the Open Government Licence (India).
2. **State crop guides (package of practices)**: PDFs you add, e.g. the West Bengal agriculture department's boro rice and potato guides, and KVK/ICAR leaflets.
3. **The Agronavis disease library**: rows of `crop_diseases` that have a description, symptoms or treatment written.

**What it does:** cleans and de-duplicates the notes, embeds them with **bge-m3** (open, multilingual: a Bengali or Hindi question finds an English note), and uploads them to the `knowledge_chunks` table (`supabase/migrations/0023_knowledge_base.sql`). The API embeds questions with the same model on Cloudflare's free tier.

**Before you run it**
- Run the migration: `npm run db:push` (or paste `0023_knowledge_base.sql` into the Supabase SQL editor).
- In Colab, open 🔑 *Secrets* and add `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY`. Never paste the service key into a cell.
- *Runtime ▸ Change runtime type ▸ T4 GPU*. About 15–30 minutes.

Run it again whenever you add documents: rows are updated, not duplicated.

In [ ]:
!pip -q install -U sentence-transformers datasets supabase pypdf

import os, re, glob, json, hashlib, datetime, collections
import pandas as pd, numpy as np, torch
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/agronavis/knowledge'
KCC_DIR    = f'{DRIVE_ROOT}/kcc'     # optional: KCC CSVs downloaded from data.gov.in
DOCS_DIR   = f'{DRIVE_ROOT}/docs'    # PDFs / .txt / .md crop guides + docs_manifest.csv
for d in (KCC_DIR, DOCS_DIR): os.makedirs(d, exist_ok=True)

# Where the farmers are. KCC answers from these states are kept.
STATES = ['WEST BENGAL', 'BIHAR', 'UTTAR PRADESH', 'JHARKHAND', 'ODISHA', 'ASSAM']
# Advice changes (and some pesticides were banned), so only recent answers.
MIN_YEAR = 2019
# Supabase's free database is 500 MB, shared with the app. About 5 KB per note
# including the vector index, so 30,000 notes is about 150 MB.
MAX_NOTES_PER_STATE = {'WEST BENGAL': 12000, 'BIHAR': 6000, 'UTTAR PRADESH': 6000,
                       'JHARKHAND': 2000, 'ODISHA': 2000, 'ASSAM': 2000}

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_SERVICE_ROLE_KEY')
assert SUPABASE_URL and SUPABASE_KEY, 'Add SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY in Colab Secrets (key icon, left).'

## 1. Kisan Call Centre answers

If you have downloaded KCC CSVs from [data.gov.in](https://www.data.gov.in/resource/kisan-call-centre-kcc-transcripts-farmers-queries-answers), put them in `MyDrive/agronavis/knowledge/kcc/`. Otherwise the notebook reads a public mirror of the same records on Hugging Face ([Omegaindebt/Kisan_Call_Centre_Transcripts](https://huggingface.co/datasets/Omegaindebt/Kisan_Call_Centre_Transcripts), about 1M rows, 2009–2024). Columns: `StateName, DistrictName, BlockName, Season, Sector, Category, Crop, QueryType, QueryText, KccAns, CreatedOn`.

In [ ]:
COLS = ['StateName', 'DistrictName', 'Season', 'Sector', 'Category', 'Crop', 'QueryType', 'QueryText', 'KccAns', 'CreatedOn']

csvs = glob.glob(f'{KCC_DIR}/**/*.csv', recursive=True)
if csvs:
    frames = []
    for f in csvs:
        df = pd.read_csv(f, dtype=str, on_bad_lines='skip')
        df.columns = [c.strip() for c in df.columns]
        frames.append(df[[c for c in COLS if c in df.columns]])
    kcc = pd.concat(frames, ignore_index=True)
    print(f'{len(kcc):,} rows from {len(csvs)} CSV file(s)')
else:
    from datasets import load_dataset
    ds = load_dataset('Omegaindebt/Kisan_Call_Centre_Transcripts', split='train')
    ds = ds.filter(lambda r: (r['StateName'] or '').upper() in STATES, num_proc=2)
    kcc = ds.to_pandas()
    print(f'{len(kcc):,} rows for {STATES} from the Hugging Face mirror')

kcc['StateName'] = kcc['StateName'].str.upper().str.strip()
kcc = kcc[kcc['StateName'].isin(STATES)]
kcc['when'] = pd.to_datetime(kcc['CreatedOn'], errors='coerce')
kcc = kcc[kcc['when'].dt.year >= MIN_YEAR]
print(f'{len(kcc):,} rows from {MIN_YEAR} on')
print(kcc['QueryType'].value_counts().head(25))

### Clean it

Dropped:
- **Time-bound questions**: weather, market rates, scheme amounts. Sahayak gets those live.
- **Non-answers** ("information given", "call transferred", a phone number).
- **Banned chemicals**: any answer naming a pesticide on India's banned/withdrawn list. The list below is a starting point; check the current list on the [CIB&RC site](https://ppqs.gov.in/divisions/cib-rc) and add to it.
- **Repeats**: the same question about the same crop in the same state is kept once, with its most recent, fullest answer. How often it was asked is kept as its popularity.

In [ ]:
DROP_TYPES = re.compile(r'weather|market|price|mandi|scheme|loan|credit|insurance|subsid|kcc|call back|contact', re.I)
NON_ANSWER = re.compile(
    r'^\s*(?:given|provided|suggested|informed|told|explained)?\s*(?:the\s+)?(?:information|info|details?|answer)\b.{0,40}$'
    r'|transferr|call (?:back|dropped|disconnected)|contact (?:no|number)|\b\d{10}\b|\bKVK no\b', re.I)
# Banned, withdrawn or restricted in India (2018 orders and earlier). Conservative on purpose; add to it.
BANNED = re.compile(r'endosulfan|monocrotophos|methyl ?parathion|phorate|phosphamidon|triazophos|dichlorvos|'
                    r'trichlorfon|alachlor|benomyl|carbaryl|diazinon|fenarimol|fenthion|linuron|thiometon|'
                    r'tridemorph|trifluralin|aldrin|dieldrin|chlordane|heptachlor|\bbhc\b|\bddt\b|lindane', re.I)

CROP_ALIAS = [('rice', r'paddy|dhan|rice'), ('wheat', r'wheat'), ('potato', r'potato'), ('jute', r'jute'),
              ('mustard', r'mustard|rapeseed|toria|sarson'), ('maize', r'maize|corn'), ('tomato', r'tomato'),
              ('brinjal', r'brinjal|eggplant'), ('onion', r'onion'), ('chilli', r'chil+i'),
              ('cauliflower', r'cauliflower'), ('cabbage', r'cabbage'), ('lentil', r'lentil|masoor'),
              ('banana', r'banana'), ('mango', r'mango'), ('sugarcane', r'sugarcane')]

def crop_name(raw):
    s = str(raw or '').lower()
    for name, pat in CROP_ALIAS:
        if re.search(pat, s): return name
    s = re.sub(r'\(.*?\)', '', s).strip()
    return None if s in ('', 'others', 'other', 'nan', 'none') else s

def clean(t):
    return re.sub(r'\s+', ' ', str(t or '')).strip(' .')

k = kcc.copy()
k['q'] = k['QueryText'].map(clean); k['a'] = k['KccAns'].map(clean)
k = k[~k['QueryType'].fillna('').str.contains(DROP_TYPES)]
k = k[(k['a'].str.len() >= 25) & (k['q'].str.len() >= 8)]
k = k[~k['a'].str.contains(NON_ANSWER)]
k = k[~(k['a'] + ' ' + k['q']).str.contains(BANNED)]
k['crop'] = k['Crop'].map(crop_name)
k['qkey'] = k['q'].str.lower().str.replace(r'[^a-z0-9ऀ-৿ ]+', ' ', regex=True).str.split().str.join(' ')
k = k.sort_values(['when', 'a'], key=lambda c: c.str.len() if c.name == 'a' else c, ascending=False)
k['asked'] = k.groupby(['StateName', 'crop', 'qkey'], dropna=False)['qkey'].transform('size')
k = k.drop_duplicates(['StateName', 'crop', 'qkey'])

parts = []
for state, cap in MAX_NOTES_PER_STATE.items():
    parts.append(k[k['StateName'] == state].sort_values('asked', ascending=False).head(cap))
k = pd.concat(parts)
print(f'{len(k):,} KCC notes kept'); print(k['StateName'].value_counts()); print(k['crop'].value_counts().head(15))

def ref(*xs): return hashlib.sha1('|'.join(map(str, xs)).encode()).hexdigest()[:20]

NOTES = [{
    'source': 'kcc',
    'source_ref': ref(r.StateName, r.crop, r.qkey),
    'title': ' · '.join(x for x in [str(r.Crop or '').title(), str(r.QueryType or '').title()] if x and x != 'Nan'),
    'url': None,
    'state': r.StateName.lower(),
    'crop': r.crop,
    'language': 'en',
    'content': f'Q: {r.q}\nA: {r.a}',
} for r in k.itertuples()]
print(NOTES[0])

## 2. State crop guides (your PDFs)

Put PDFs, `.txt` or `.md` files in `MyDrive/agronavis/knowledge/docs/`. Add a `docs_manifest.csv` beside them so each note can say where it came from:

```
file,title,url,state,crop
boro_rice_pop.pdf,West Bengal boro rice package of practices,https://…,west bengal,rice
potato_late_blight.pdf,Late blight of potato (KVK leaflet),,west bengal,potato
```
Leave `state` empty for guides that apply everywhere. Good first files: the West Bengal agriculture department's package of practices for boro and aman rice, potato, jute and mustard, and ICAR/KVK advisories for your district.

In [ ]:
from pypdf import PdfReader

manifest = {}
mf = f'{DOCS_DIR}/docs_manifest.csv'
if os.path.exists(mf):
    for row in pd.read_csv(mf, dtype=str).fillna('').to_dict('records'):
        manifest[row['file']] = row

def text_of(path):
    if path.lower().endswith('.pdf'):
        return '\n'.join((p.extract_text() or '') for p in PdfReader(path).pages)
    return open(path, encoding='utf-8', errors='ignore').read()

def chunks(text, words=220, overlap=40):
    w = re.sub(r'\s+', ' ', text).split()
    step = words - overlap
    for i in range(0, max(len(w) - overlap, 1), step):
        piece = ' '.join(w[i:i + words])
        if len(piece) > 200: yield i // step, piece

doc_notes = []
for path in sorted(glob.glob(f'{DOCS_DIR}/**/*', recursive=True)):
    if not path.lower().endswith(('.pdf', '.txt', '.md')): continue
    name = os.path.basename(path); meta = manifest.get(name, {})
    try:
        text = text_of(path)
    except Exception as e:
        print('could not read', name, e); continue
    for i, piece in chunks(text):
        doc_notes.append({
            'source': 'pop', 'source_ref': f'{name}#{i}',
            'title': meta.get('title') or os.path.splitext(name)[0].replace('_', ' '),
            'url': meta.get('url') or None,
            'state': (meta.get('state') or '').lower() or None,
            'crop': crop_name(meta.get('crop')) if meta.get('crop') else None,
            'language': 'en', 'content': piece,
        })
print(f'{len(doc_notes)} chunks from documents')
if not doc_notes:
    print('No documents yet. That is fine: KCC alone is a good start.')

## 3. The Agronavis disease library

Only rows someone has written content for. Rows that are just a class name add nothing.

In [ ]:
from supabase import create_client
sb = create_client(SUPABASE_URL, SUPABASE_KEY)

rows = sb.table('crop_diseases').select('class_key,name,crop_type,description,symptoms,treatment').execute().data
lib_notes = []
for r in rows:
    parts = [r.get('description') or '']
    if r.get('symptoms'): parts.append('Symptoms: ' + '; '.join(r['symptoms']))
    if r.get('treatment'): parts.append('Management: ' + '; '.join(r['treatment']))
    body = ' '.join(p for p in parts if p).strip()
    if len(body) < 40: continue
    lib_notes.append({'source': 'disease_library', 'source_ref': r['class_key'],
                      'title': f"{r['crop_type']} · {r['name']}", 'url': None, 'state': None,
                      'crop': crop_name(r['crop_type']), 'language': 'en',
                      'content': f"{r['crop_type']} — {r['name']}. {body}"})
print(f'{len(lib_notes)} disease library notes')

## 4. Embed with bge-m3

The same open model the API uses for questions (on Cloudflare), so the vectors match.

In [ ]:
from sentence_transformers import SentenceTransformer

ALL = NOTES + doc_notes + lib_notes
print(f'{len(ALL):,} notes to embed')
model = SentenceTransformer('BAAI/bge-m3', device='cuda')
model.max_seq_length = 512
texts = [(n['title'] + '\n' if n['title'] else '') + n['content'] for n in ALL]
vecs = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True,
                    convert_to_numpy=True)
assert vecs.shape[1] == 1024, vecs.shape

## 5. Upload to Supabase

Upserted on `(source, source_ref)`, so a re-run updates notes instead of duplicating them.

In [ ]:
def as_halfvec(v):
    return '[' + ','.join(f'{x:.5f}' for x in v) + ']'

BATCH = 300
for i in range(0, len(ALL), BATCH):
    batch = [{**n, 'embedding': as_halfvec(v)} for n, v in zip(ALL[i:i + BATCH], vecs[i:i + BATCH])]
    sb.table('knowledge_chunks').upsert(batch, on_conflict='source,source_ref').execute()
    if (i // BATCH) % 10 == 0:
        print(f'{i + len(batch):,} / {len(ALL):,}')
print('done:', sb.table('knowledge_chunks').select('id', count='exact').limit(1).execute().count, 'notes in the table')

## 6. Try it

The same search the API runs. Ask in Bengali, Hindi or English.

In [ ]:
def ask(q, state='west bengal'):
    v = model.encode([q], normalize_embeddings=True)[0]
    res = sb.rpc('match_knowledge', {'query_text': q, 'query_embedding': as_halfvec(v),
                                     'filter_state': state, 'match_count': 5}).execute().data
    print(f'\n❓ {q}')
    for r in res:
        print(f"  [{r['source']}·{r.get('crop')}] {r['content'][:180]}…")

ask('ধানের পাতায় বাদামী দাগ, কী করব?')           # Bengali: brown spots on paddy leaves
ask('आलू में झुलसा रोग से कैसे बचाएं?', 'bihar')   # Hindi: protecting potato from blight
ask('When to drain boro rice field before harvest')